# Tutorial 15 — Multimodal system integration, research critique, and project defence

[Lecture notes](../lectures/15_capstone.md) · [Exercise sheet](../exercises/15_capstone.md)

**40 minutes, CPU, no accounts or model downloads.** Requires Python 3.10+ and NumPy. These are mechanism experiments using authored or synthetic data, not frontier benchmark results. Restart the kernel and run all cells in order.

**Objective:** Run a text-plus-chart evidence pipeline under modality ablations, calculate paired uncertainty, and write reproducible result records.

Allocate 5 minutes to setup, 15 to the mechanism, 10 to interventions, and 10 to discussion. Record predictions before running. Complete the separate exercise sheet after class.


In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)


Python 3.13.1 NumPy 2.1.3 seed 42


## 1. Create paired policy and visual evidence

Every example needs a policy selecting A/B/C and chart pixels specifying values. Data are authored synthetic cases. No model inference is hidden in this baseline.

Before running: predict the output or the invariant being checked.


In [2]:
import json

def render(values):
    im=np.full((48,72,3),255,dtype=np.uint8)
    for j,h in enumerate(values): im[40-int(h)*4:40,4+24*j:16+24*j]=[30,60,90]
    return im

def read(im,j):
    return int((im[:40,8+24*j,0]<100).sum()/4)
cases=[]
for i in range(12):
    values=rng.integers(1,10,size=3); chosen=int(rng.integers(0,3))
    cases.append({"id":f"case-{i}","split":"test","policy":f"Use plot {'ABC'[chosen]}.","image":render(values),"gold":int(values[chosen])})
def system(case,use_text=True,use_image=True):
    if not use_text or not use_image: return {"answer":None,"evidence_ids":[]}
    chosen="ABC".index(case["policy"].split()[-1][0])
    return {"answer":read(case["image"],chosen),"evidence_ids":[case["id"]+"-policy",case["id"]+"-chart"]}
print(cases[0]["policy"],system(cases[0]),"gold",cases[0]["gold"])


Use plot B. {'answer': 7, 'evidence_ids': ['case-0-policy', 'case-0-chart']} gold 7


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 2. Run modality ablations

Accuracy counts abstentions as incorrect here; selective accuracy and coverage explain abstention behavior. These test cases demonstrate mechanics, not deployment readiness.

Before running: predict the output or the invariant being checked.


In [3]:
conditions={"full":(True,True),"text_only":(True,False),"image_only":(False,True)}
results={}
for name,(text_on,image_on) in conditions.items():
    predictions=[system(c,text_on,image_on) for c in cases]
    correct=np.array([p["answer"]==c["gold"] for p,c in zip(predictions,cases)])
    answered=np.array([p["answer"] is not None for p in predictions])
    results[name]=correct
    print(name,{"accuracy":float(correct.mean()),"coverage":float(answered.mean()),"selective_accuracy":float(correct[answered].mean()) if answered.any() else None})
assert results["full"].all() and not results["text_only"].any()
idx=rng.integers(0,len(cases),size=(1000,len(cases)))
difference=(results["full"].astype(float)-results["image_only"].astype(float))[idx].mean(axis=1)
print("paired interval full minus image-only",np.quantile(difference,[.025,.975]))


full {'accuracy': 1.0, 'coverage': 1.0, 'selective_accuracy': 1.0}
text_only {'accuracy': 0.0, 'coverage': 0.0, 'selective_accuracy': None}
image_only {'accuracy': 0.0, 'coverage': 0.0, 'selective_accuracy': None}
paired interval full minus image-only [1. 1.]


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 3. Record provenance, not fabricated timings

Runtime is null because it was not measured. A degenerate interval on these deterministic toy cases cannot establish real-world certainty.

Before running: predict the output or the invariant being checked.


In [4]:
records=[]
for c in cases:
    p=system(c)
    row={"id":c["id"],"condition":"full","split":c["split"],"language":"en","modalities":["text","image"],"prediction":p["answer"],"target":c["gold"],"evidence_ids":p["evidence_ids"],"model":"deterministic-chart-policy-v1","revision":"course-authored-v1","prompt_version":"policy-v1","seed":42,"latency_ms":None}
    assert len(row["evidence_ids"])==2
    records.append(row)
print(json.dumps(records[0],indent=2))
# Opt-in export: uncomment after choosing an output path.
# from pathlib import Path
# Path("capstone-results.jsonl").write_text("\n".join(json.dumps(r) for r in records)+"\n")


{
  "id": "case-0",
  "condition": "full",
  "split": "test",
  "language": "en",
  "modalities": [
    "text",
    "image"
  ],
  "prediction": 7,
  "target": 7,
  "evidence_ids": [
    "case-0-policy",
    "case-0-chart"
  ],
  "model": "deterministic-chart-policy-v1",
  "revision": "course-authored-v1",
  "prompt_version": "policy-v1",
  "seed": 42,
  "latency_ms": null
}


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## Practical discussion

1. What experiment distinguishes modality use from a memorized answer?
2. Why are null and zero different for an unmeasured runtime?
3. What makes a negative result useful?


## Extension and submission

Choose one real-model extension from earlier days for the final project. Keep the deterministic baseline and identical test protocol so the added model can be evaluated fairly.

See the [extension guide](../extensions/README.md) before running model downloads. Extensions require additional dependencies and are not necessary for full exercise credit.

Submit the completed notebook with a 150–250-word interpretation and the separate exercise answers. Include a baseline, one controlled change, data provenance, seed, and an honest limitation. Never replace measured results with expected trends.
